# MDM Organization Manager

Interactive UI for viewing, searching, and editing organization data from `mdm_genai_assets.demoui.orgtable`.

**Features:**
- Smart search on organization name and ID
- Paginated data table with sorting
- Inline editing with save functionality

Run all cells below to launch the UI.

In [ ]:
# Cell 1: Setup & Configuration
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import re
from datetime import datetime

# Table configuration
TABLE_NAME = "mdm_genai_assets.demoui.orgtable"
PAGE_SIZE = 25

# Columns to display in the results table
DISPLAY_COLUMNS = [
    "organization_id",
    "organization_name",
    "organization_status",
    "bed_count",
    "source_system",
    "parent_organization_name",
    "genai_organization_name",
    "genai_primary_specialty",
]

# Columns that users can edit
EDITABLE_COLUMNS = [
    "organization_name",
    "organization_status",
    "bed_count",
    "is_subpart",
    "parent_organization_name",
    "genai_organization_name",
    "genai_primary_specialty",
    "genai_citation",
]

# All columns in the table
ALL_COLUMNS = [
    "organization_id", "organization_name", "organization_status",
    "organization_status_date", "bed_count", "is_subpart",
    "parent_organization_name", "enumeration_date", "last_update_date",
    "source_system", "source_record_id", "genai_organization_name",
    "genai_primary_specialty", "genai_citation", "_run_cycle",
    "_run_date", "_record_hash", "created_date", "updated_date",
]

# Status options for dropdown
STATUS_OPTIONS = ["A", "I", "D"]

print("Configuration loaded successfully.")
print(f"Table: {TABLE_NAME}")
print(f"Page size: {PAGE_SIZE}")

In [ ]:
# Cell 2: Database Helper Functions

def sanitize_input(value):
    """Escape single quotes and special characters for safe SQL embedding."""
    if value is None:
        return None
    return str(value).replace("'", "''").replace("\\", "\\\\")


def build_search_where_clause(query):
    """Build WHERE clause for multi-token search across name and ID."""
    if not query or not query.strip():
        return ""
    
    tokens = query.strip().split()
    conditions = []
    for token in tokens:
        safe_token = sanitize_input(token.lower())
        conditions.append(
            f"(LOWER(organization_name) LIKE '%{safe_token}%' "
            f"OR LOWER(organization_id) LIKE '%{safe_token}%')"
        )
    return "WHERE " + " AND ".join(conditions)


def build_relevance_order(query):
    """Build ORDER BY clause with relevance scoring."""
    if not query or not query.strip():
        return "ORDER BY organization_name ASC"
    
    safe_query = sanitize_input(query.strip().lower())
    return (
        f"ORDER BY CASE "
        f"WHEN LOWER(organization_name) = '{safe_query}' THEN 0 "
        f"WHEN LOWER(organization_id) = '{safe_query}' THEN 1 "
        f"WHEN LOWER(organization_name) LIKE '{safe_query}%' THEN 2 "
        f"WHEN LOWER(organization_id) LIKE '{safe_query}%' THEN 3 "
        f"ELSE 4 END, organization_name ASC"
    )


def search_organizations(query="", page=0, page_size=PAGE_SIZE):
    """Search organizations with pagination. Returns (pandas_df, total_count)."""
    where_clause = build_search_where_clause(query)
    order_clause = build_relevance_order(query)
    offset = page * page_size
    
    # Get total count
    count_sql = f"SELECT COUNT(*) as cnt FROM {TABLE_NAME} {where_clause}"
    total_count = spark.sql(count_sql).collect()[0]["cnt"]
    
    # Get page of results
    cols = ", ".join(ALL_COLUMNS)
    data_sql = (
        f"SELECT {cols} FROM {TABLE_NAME} "
        f"{where_clause} {order_clause} "
        f"LIMIT {page_size} OFFSET {offset}"
    )
    df = spark.sql(data_sql).toPandas()
    
    return df, total_count


def get_organization(org_id):
    """Fetch a single organization by ID. Returns a dict or None."""
    safe_id = sanitize_input(org_id)
    sql = f"SELECT * FROM {TABLE_NAME} WHERE organization_id = '{safe_id}'"
    df = spark.sql(sql).toPandas()
    if df.empty:
        return None
    return df.iloc[0].to_dict()


def update_organization(org_id, field_updates):
    """Update specific fields for an organization. Returns True on success."""
    if not field_updates:
        return False
    
    safe_id = sanitize_input(org_id)
    set_clauses = []
    
    for field, value in field_updates.items():
        if field not in EDITABLE_COLUMNS:
            continue
        if value is None or (isinstance(value, str) and value.strip() == ""):
            set_clauses.append(f"{field} = NULL")
        elif field == "bed_count":
            try:
                int_val = int(value)
                set_clauses.append(f"{field} = {int_val}")
            except (ValueError, TypeError):
                set_clauses.append(f"{field} = NULL")
        elif field == "is_subpart":
            bool_val = str(value).lower() in ("true", "1", "yes", "y")
            set_clauses.append(f"{field} = {str(bool_val).lower()}")
        else:
            safe_val = sanitize_input(value)
            set_clauses.append(f"{field} = '{safe_val}'")
    
    if not set_clauses:
        return False
    
    set_clauses.append("updated_date = current_timestamp()")
    set_str = ", ".join(set_clauses)
    
    sql = f"UPDATE {TABLE_NAME} SET {set_str} WHERE organization_id = '{safe_id}'"
    spark.sql(sql)
    return True


print("Database helper functions loaded.")

In [ ]:
# Cell 3: HTML Table Renderer

TABLE_CSS = """
<style>
.mdm-table-container {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    max-width: 100%;
    overflow-x: auto;
}
.mdm-header {
    background: linear-gradient(135deg, #1a73e8, #0d47a1);
    color: white;
    padding: 16px 24px;
    border-radius: 8px 8px 0 0;
    font-size: 20px;
    font-weight: 600;
}
.mdm-stats {
    background: #f8f9fa;
    padding: 10px 24px;
    color: #5f6368;
    font-size: 13px;
    border-bottom: 1px solid #e0e0e0;
}
.mdm-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}
.mdm-table th {
    background: #f1f3f4;
    color: #202124;
    padding: 10px 12px;
    text-align: left;
    font-weight: 600;
    border-bottom: 2px solid #dadce0;
    white-space: nowrap;
}
.mdm-table td {
    padding: 8px 12px;
    border-bottom: 1px solid #e8eaed;
    max-width: 250px;
    overflow: hidden;
    text-overflow: ellipsis;
    white-space: nowrap;
}
.mdm-table tr:nth-child(even) {
    background: #fafbfc;
}
.mdm-table tr:hover {
    background: #e8f0fe;
}
.mdm-highlight {
    background-color: #fdd835;
    padding: 1px 2px;
    border-radius: 2px;
    font-weight: 500;
}
.mdm-status-active {
    background: #e6f4ea;
    color: #137333;
    padding: 3px 10px;
    border-radius: 12px;
    font-size: 12px;
    font-weight: 500;
}
.mdm-status-inactive {
    background: #fce8e6;
    color: #c5221f;
    padding: 3px 10px;
    border-radius: 12px;
    font-size: 12px;
    font-weight: 500;
}
.mdm-status-deactivated {
    background: #f1f3f4;
    color: #5f6368;
    padding: 3px 10px;
    border-radius: 12px;
    font-size: 12px;
    font-weight: 500;
}
.mdm-no-results {
    text-align: center;
    padding: 40px;
    color: #5f6368;
    font-size: 16px;
}
</style>
"""


def highlight_text(text, query):
    """Highlight search query matches in text."""
    if not query or not text:
        return str(text) if text is not None else ""
    text = str(text)
    tokens = query.strip().split()
    for token in tokens:
        pattern = re.compile(re.escape(token), re.IGNORECASE)
        text = pattern.sub(lambda m: f'<span class="mdm-highlight">{m.group()}</span>', text)
    return text


def format_status(status):
    """Format organization status as a colored badge."""
    if not status:
        return ""
    s = str(status).upper()
    if s == "A":
        return '<span class="mdm-status-active">Active</span>'
    elif s == "I":
        return '<span class="mdm-status-inactive">Inactive</span>'
    elif s == "D":
        return '<span class="mdm-status-deactivated">Deactivated</span>'
    return f'<span class="mdm-status-deactivated">{status}</span>'


def render_html_table(df, search_query="", page=0, total_count=0, page_size=PAGE_SIZE):
    """Render a styled HTML table from a pandas DataFrame."""
    start = page * page_size + 1
    end = min(start + len(df) - 1, total_count)
    total_pages = max(1, (total_count + page_size - 1) // page_size)
    
    search_info = f' for \"{search_query}\"' if search_query else ""
    
    html = TABLE_CSS
    html += '<div class="mdm-table-container">'
    html += '<div class="mdm-header">MDM Organization Manager</div>'
    html += f'<div class="mdm-stats">Showing {start}-{end} of {total_count:,} results{search_info} &nbsp;|&nbsp; Page {page + 1} of {total_pages}</div>'
    
    if df.empty:
        html += '<div class="mdm-no-results">No organizations found. Try a different search term.</div>'
        html += '</div>'
        return html
    
    html += '<table class="mdm-table">'
    
    # Header row
    html += "<thead><tr>"
    html += '<th style="width: 30px;">#</th>'
    for col in DISPLAY_COLUMNS:
        label = col.replace("_", " ").title()
        html += f"<th>{label}</th>"
    html += "</tr></thead>"
    
    # Data rows
    html += "<tbody>"
    for idx, row in df.iterrows():
        row_num = start + df.index.get_loc(idx) if hasattr(df.index, 'get_loc') else start + idx
        html += "<tr>"
        html += f'<td style="color: #9aa0a6;">{row_num}</td>'
        
        for col in DISPLAY_COLUMNS:
            val = row.get(col, "")
            if col == "organization_status":
                html += f"<td>{format_status(val)}</td>"
            elif col in ("organization_name", "organization_id"):
                html += f"<td>{highlight_text(val, search_query)}</td>"
            elif col == "bed_count":
                display_val = int(val) if val is not None and str(val) != "None" and str(val) != "nan" else "-"
                html += f"<td>{display_val}</td>"
            else:
                display_val = str(val) if val is not None and str(val) != "None" and str(val) != "nan" else "-"
                html += f'<td title="{display_val}">{display_val}</td>'
        
        html += "</tr>"
    
    html += "</tbody></table></div>"
    return html


print("HTML table renderer loaded.")

In [ ]:
# Cell 4: Edit Form Builder

def build_edit_form(org_id, on_save_callback, on_cancel_callback):
    """Build an ipywidgets form for editing an organization record."""
    org = get_organization(org_id)
    if org is None:
        return widgets.HTML(value=f'<p style="color: red;">Organization {org_id} not found.</p>')
    
    # Title
    title = widgets.HTML(
        value=(
            f'<div style="background: linear-gradient(135deg, #1a73e8, #0d47a1); '
            f'color: white; padding: 16px 24px; border-radius: 8px 8px 0 0; '
            f'font-size: 18px; font-weight: 600;">'
            f'Editing: {org_id}</div>'
        )
    )
    
    # Read-only info
    readonly_info = widgets.HTML(
        value=(
            f'<div style="background: #f8f9fa; padding: 10px 24px; '
            f'font-size: 12px; color: #5f6368; border-bottom: 1px solid #e0e0e0;">'
            f'<b>Source:</b> {org.get("source_system", "-")} &nbsp;|&nbsp; '
            f'<b>Source Record ID:</b> {org.get("source_record_id", "-")} &nbsp;|&nbsp; '
            f'<b>Created:</b> {org.get("created_date", "-")} &nbsp;|&nbsp; '
            f'<b>Last Updated:</b> {org.get("updated_date", "-")} &nbsp;|&nbsp; '
            f'<b>Enumeration Date:</b> {org.get("enumeration_date", "-")}'
            f'</div>'
        )
    )
    
    # Editable fields
    style = {'description_width': '180px'}
    layout = widgets.Layout(width='600px')
    
    name_input = widgets.Text(
        value=str(org.get("organization_name", "") or ""),
        description="Organization Name:",
        style=style, layout=layout
    )
    
    status_val = str(org.get("organization_status", "") or "")
    status_options = STATUS_OPTIONS if status_val in STATUS_OPTIONS else [status_val] + STATUS_OPTIONS
    status_input = widgets.Dropdown(
        options=status_options,
        value=status_val if status_val else STATUS_OPTIONS[0],
        description="Status:",
        style=style, layout=layout
    )
    
    bed_val = org.get("bed_count", None)
    bed_count_input = widgets.Text(
        value=str(int(bed_val)) if bed_val is not None and str(bed_val) not in ("None", "nan") else "",
        description="Bed Count:",
        style=style, layout=layout
    )
    
    is_subpart_val = org.get("is_subpart", None)
    is_subpart_input = widgets.Dropdown(
        options=[("Yes", True), ("No", False), ("Unknown", None)],
        value=is_subpart_val if is_subpart_val in (True, False) else None,
        description="Is Subpart:",
        style=style, layout=layout
    )
    
    parent_org_input = widgets.Text(
        value=str(org.get("parent_organization_name", "") or ""),
        description="Parent Org Name:",
        style=style, layout=layout
    )
    
    genai_name_input = widgets.Text(
        value=str(org.get("genai_organization_name", "") or ""),
        description="GenAI Org Name:",
        style=style, layout=layout
    )
    
    genai_specialty_input = widgets.Text(
        value=str(org.get("genai_primary_specialty", "") or ""),
        description="GenAI Specialty:",
        style=style, layout=layout
    )
    
    genai_citation_input = widgets.Textarea(
        value=str(org.get("genai_citation", "") or ""),
        description="GenAI Citation:",
        style=style,
        layout=widgets.Layout(width='600px', height='80px')
    )
    
    # Status message area
    status_output = widgets.Output()
    
    # Buttons
    save_btn = widgets.Button(
        description='Save Changes',
        button_style='success',
        icon='check',
        layout=widgets.Layout(width='150px')
    )
    
    cancel_btn = widgets.Button(
        description='Cancel',
        button_style='warning',
        icon='times',
        layout=widgets.Layout(width='120px')
    )
    
    back_btn = widgets.Button(
        description='Back to Results',
        button_style='info',
        icon='arrow-left',
        layout=widgets.Layout(width='160px')
    )
    
    def on_save_click(_):
        updates = {}
        if name_input.value != str(org.get("organization_name", "") or ""):
            updates["organization_name"] = name_input.value
        if status_input.value != str(org.get("organization_status", "") or ""):
            updates["organization_status"] = status_input.value
        
        old_bed = str(int(org.get("bed_count", 0) or 0)) if org.get("bed_count") is not None and str(org.get("bed_count")) not in ("None", "nan") else ""
        if bed_count_input.value != old_bed:
            updates["bed_count"] = bed_count_input.value if bed_count_input.value else None
        
        if is_subpart_input.value != org.get("is_subpart", None):
            updates["is_subpart"] = is_subpart_input.value
        if parent_org_input.value != str(org.get("parent_organization_name", "") or ""):
            updates["parent_organization_name"] = parent_org_input.value
        if genai_name_input.value != str(org.get("genai_organization_name", "") or ""):
            updates["genai_organization_name"] = genai_name_input.value
        if genai_specialty_input.value != str(org.get("genai_primary_specialty", "") or ""):
            updates["genai_primary_specialty"] = genai_specialty_input.value
        if genai_citation_input.value != str(org.get("genai_citation", "") or ""):
            updates["genai_citation"] = genai_citation_input.value
        
        with status_output:
            clear_output()
            if not updates:
                display(HTML('<p style="color: #f9a825;">No changes detected.</p>'))
                return
            try:
                success = update_organization(org_id, updates)
                if success:
                    changed_fields = ", ".join(updates.keys())
                    display(HTML(
                        f'<p style="color: #137333; font-weight: 500;">'
                        f'Successfully updated {changed_fields} for {org_id}.</p>'
                    ))
                    on_save_callback()
                else:
                    display(HTML('<p style="color: #c5221f;">Update failed. No valid fields to update.</p>'))
            except Exception as e:
                display(HTML(f'<p style="color: #c5221f;">Error: {str(e)}</p>'))
    
    def on_cancel_click(_):
        on_cancel_callback()
    
    save_btn.on_click(on_save_click)
    cancel_btn.on_click(on_cancel_click)
    back_btn.on_click(on_cancel_click)
    
    form = widgets.VBox([
        title,
        readonly_info,
        back_btn,
        widgets.HTML(value='<div style="height: 12px;"></div>'),
        name_input,
        status_input,
        bed_count_input,
        is_subpart_input,
        parent_org_input,
        genai_name_input,
        genai_specialty_input,
        genai_citation_input,
        widgets.HTML(value='<div style="height: 8px;"></div>'),
        widgets.HBox([save_btn, cancel_btn]),
        status_output,
    ])
    
    return form


print("Edit form builder loaded.")

In [ ]:
# Cell 5: Main UI Application

# --- Application State ---
class AppState:
    def __init__(self):
        self.current_query = ""
        self.current_page = 0
        self.total_count = 0
        self.current_df = None

state = AppState()

# --- UI Widgets ---

# Search bar
search_input = widgets.Text(
    placeholder='Search by organization name or ID...',
    layout=widgets.Layout(width='450px'),
    style={'description_width': '0px'}
)

search_btn = widgets.Button(
    description='Search',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='110px')
)

clear_btn = widgets.Button(
    description='Clear',
    button_style='',
    icon='eraser',
    layout=widgets.Layout(width='90px')
)

refresh_btn = widgets.Button(
    description='Refresh',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='110px')
)

search_bar = widgets.HBox(
    [search_input, search_btn, clear_btn, refresh_btn],
    layout=widgets.Layout(margin='10px 0')
)

# Pagination controls
prev_btn = widgets.Button(
    description='Previous',
    button_style='',
    icon='arrow-left',
    layout=widgets.Layout(width='120px')
)

next_btn = widgets.Button(
    description='Next',
    button_style='',
    icon='arrow-right',
    layout=widgets.Layout(width='120px')
)

page_label = widgets.HTML(value='<span style="padding: 0 16px; font-size: 14px;">Page 1</span>')

pagination_bar = widgets.HBox(
    [prev_btn, page_label, next_btn],
    layout=widgets.Layout(margin='8px 0', justify_content='center')
)

# Edit row selector
edit_id_input = widgets.Text(
    placeholder='Enter Organization ID to edit...',
    layout=widgets.Layout(width='350px'),
    style={'description_width': '0px'}
)

edit_btn = widgets.Button(
    description='Edit Record',
    button_style='warning',
    icon='pencil',
    layout=widgets.Layout(width='140px')
)

edit_bar = widgets.HBox(
    [widgets.HTML(value='<span style="font-size: 13px; padding: 6px 8px 0 0; color: #5f6368;">Edit:</span>'),
     edit_id_input, edit_btn],
    layout=widgets.Layout(margin='4px 0')
)

# Output areas
results_output = widgets.Output()
edit_output = widgets.Output()
status_output = widgets.Output()

# --- Event Handlers ---

def refresh_results():
    """Fetch and display search results."""
    with status_output:
        clear_output()
        display(HTML('<span style="color: #1a73e8;">Loading...</span>'))
    
    try:
        df, total = search_organizations(state.current_query, state.current_page, PAGE_SIZE)
        state.current_df = df
        state.total_count = total
        
        total_pages = max(1, (total + PAGE_SIZE - 1) // PAGE_SIZE)
        page_label.value = f'<span style="padding: 0 16px; font-size: 14px;">Page {state.current_page + 1} of {total_pages}</span>'
        
        prev_btn.disabled = state.current_page <= 0
        next_btn.disabled = state.current_page >= total_pages - 1
        
        html = render_html_table(df, state.current_query, state.current_page, total, PAGE_SIZE)
        
        with results_output:
            clear_output(wait=True)
            display(HTML(html))
        
        with status_output:
            clear_output()
    
    except Exception as e:
        with status_output:
            clear_output()
            display(HTML(f'<span style="color: #c5221f;">Error: {str(e)}</span>'))


def on_search(_):
    """Handle search button click."""
    state.current_query = search_input.value.strip()
    state.current_page = 0
    # Show results view, hide edit view
    results_output.layout.display = ''
    pagination_bar.layout.display = ''
    edit_bar.layout.display = ''
    edit_output.layout.display = 'none'
    refresh_results()


def on_search_submit(change):
    """Handle Enter key in search input."""
    on_search(None)


def on_clear(_):
    """Handle clear button click."""
    search_input.value = ""
    state.current_query = ""
    state.current_page = 0
    refresh_results()


def on_prev(_):
    """Handle previous page."""
    if state.current_page > 0:
        state.current_page -= 1
        refresh_results()


def on_next(_):
    """Handle next page."""
    total_pages = max(1, (state.total_count + PAGE_SIZE - 1) // PAGE_SIZE)
    if state.current_page < total_pages - 1:
        state.current_page += 1
        refresh_results()


def on_edit(_):
    """Handle edit button click."""
    org_id = edit_id_input.value.strip()
    if not org_id:
        with status_output:
            clear_output()
            display(HTML('<span style="color: #f9a825;">Please enter an Organization ID to edit.</span>'))
        return
    
    # Hide results, show edit form
    results_output.layout.display = 'none'
    pagination_bar.layout.display = 'none'
    edit_bar.layout.display = 'none'
    
    def on_save_done():
        pass  # Stay on edit form, success message shown in form
    
    def on_cancel_done():
        # Show results, hide edit
        edit_output.layout.display = 'none'
        results_output.layout.display = ''
        pagination_bar.layout.display = ''
        edit_bar.layout.display = ''
        refresh_results()
    
    form = build_edit_form(org_id, on_save_done, on_cancel_done)
    
    with edit_output:
        clear_output(wait=True)
        display(form)
    
    edit_output.layout.display = ''


# Wire up event handlers
search_btn.on_click(on_search)
search_input.on_submit(on_search_submit)
clear_btn.on_click(on_clear)
refresh_btn.on_click(lambda _: refresh_results())
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
edit_btn.on_click(on_edit)

# Initially hide edit area
edit_output.layout.display = 'none'

# --- Main Layout ---
main_layout = widgets.VBox([
    widgets.HTML(value=(
        '<div style="font-size: 12px; color: #5f6368; margin-bottom: 4px;">'
        'Search organizations by name or ID. Use the Edit section below to modify records.'
        '</div>'
    )),
    search_bar,
    edit_bar,
    status_output,
    results_output,
    pagination_bar,
    edit_output,
])

print("UI components loaded. Run the next cell to launch.")

In [ ]:
# Cell 6: Launch the UI
# Display the main layout and load initial data

display(main_layout)

# Load initial data
refresh_results()